# AIC 2026 — Encode query vectors · SigLIP 2 + BEiT-3

Notebook chạy độc lập trên **Google Colab** hoặc **Kaggle** để biến câu query thành
vector bằng đúng model và revision đã dùng khi ingest ảnh. Việc tìm kiếm vẫn chạy trên
máy local với index đã build; máy local không cần tải model.

**Input** (tải lên runtime):

- `queries.csv`: cột `query_id`, `query_vi`, `query_en` (và `objects` nếu có). File gán
  nhãn theo `docs/evaluation/queries-template.csv` dùng được trực tiếp; `query_id` lặp
  lại chỉ được encode một lần.
- `manifest.json`: lấy từ `indexes/dev/manifest.json` trên máy local. Notebook đọc model
  và revision từ file này để vector khớp với index.

**Output:** `query_vectors.npz`, tải về máy rồi chạy:

```powershell
uv run python scripts/search_index.py --query-vectors query_vectors.npz --query-id q1
uv run python scripts/evaluate_retrieval.py --queries queries.csv --query-vectors query_vectors.npz
```

Model đều public nên không cần `HF_TOKEN`. GPU giúp nhanh hơn nhưng không bắt buộc.
Colab: tải file lên bằng thanh Files bên trái (hoặc hộp thoại hiện ra khi chạy).
Kaggle: thêm hai file vào một Dataset rồi gắn vào notebook.

In [ ]:
# transformers 4.57.1 matches the SigLIP 2 ingest notebook; timm 0.9.16 matches BEiT-3.
import importlib.metadata
import subprocess
import sys


def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


REQUIRED = {"transformers": "4.57.1", "timm": "0.9.16"}
missing = [f"{name}=={version}" for name, version in REQUIRED.items() if installed_version(name) != version]
if installed_version("sentencepiece") is None:
    missing.append("sentencepiece")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print({name: installed_version(name) for name in ("torch", "transformers", "timm", "sentencepiece")})

In [ ]:
import gc
import hashlib
import json
import os
import sys
import tarfile
import types
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch

QUERIES_CSV = "queries.csv"
MANIFEST_JSON = "manifest.json"
OUTPUT_NPZ = "query_vectors.npz"
ENCODE = ("siglip", "beit3")
SIGLIP_TEXT = "en"  # English queries for both encoders; "vi" encodes query_vi for SigLIP 2.
FORMAT_VERSION = 1

SIGLIP_MAX_LENGTH = 64
BEIT3_MAX_LENGTH = 64
BEIT3_SOURCE_REVISION = "ca43e4cd19445a536f133bf2bc25b573b2f0c7c5"
BEIT3_SOURCE_SHA256 = {
    "modeling_utils.py": "cc04b762b2c0ba32eb82d65a5e543adada156ce3584a351b56c87eb3de293a27",
    "modeling_finetune.py": "dfa9a2dbba7e7b2a46023da0d693f6e781d946d86b2b7a52e69ed4bd463d763f",
}
BEIT3_CHECKPOINT_URL = (
    "https://github.com/addf400/files/releases/download/beit3/"
    "beit3_large_patch16_384_coco_retrieval.pth"
)
BEIT3_CHECKPOINT_SIZE = 1_350_590_595
TORCHSCALE_URL = (
    "https://files.pythonhosted.org/packages/d5/7f/"
    "2a20aeaf3264eb41750b882fc56a67d907042dcf3f84745331ee4e6e7636/torchscale-0.2.0.tar.gz"
)
TORCHSCALE_SHA256 = "ca0a3982d799d6b91320a538a6cc17ea9ccb6370231934805fae78f318fe4d2f"
BEIT3_TOKENIZER_URL = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"


def runtime_name():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


RUNTIME = runtime_name()
WORK = (
    Path("/content") if RUNTIME == "colab"
    else Path("/kaggle/working") if RUNTIME == "kaggle"
    else Path.cwd()
)
MODEL_ROOT = WORK / "query-encoders"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)


def find_input(name):
    """Accept a full path, a bare file name, or (if unique) any file with the same suffix."""
    given = Path(name)
    if given.is_file():
        return given
    name = given.name  # rglob only takes relative patterns
    candidates = [WORK / name, Path.cwd() / name]
    if RUNTIME == "kaggle":
        candidates += sorted(Path("/kaggle/input").rglob(name))
    for path in candidates:
        if path.is_file():
            return path
    # Accept any file name when the inputs hold exactly one file of that type
    # (e.g. sotuyen1-queries.csv). Only the top level of the working folder is searched:
    # Colab's /content/sample_data holds unrelated CSV/JSON files.
    suffix = Path(name).suffix
    matches = {path for path in WORK.glob(f"*{suffix}") if path.is_file()}
    if RUNTIME == "kaggle":
        matches |= {path for path in Path("/kaggle/input").rglob(f"*{suffix}") if path.is_file()}
    matches = sorted(matches)
    if len(matches) == 1:
        print(f"Using {matches[0]} as {name}")
        return matches[0]
    if len(matches) > 1:
        raise FileNotFoundError(
            f"{name} not found and several {suffix} files exist; set the name in this cell: "
            + ", ".join(str(path) for path in matches)
        )
    if RUNTIME == "colab":
        from google.colab import files

        print(f"Upload the {suffix} file for {name}")
        uploaded = [Path(key) for key in files.upload()]
        picked = [path for path in uploaded if path.suffix == suffix]
        if len(picked) == 1:
            return picked[0].resolve()
    raise FileNotFoundError(f"{name} not found; upload it to {WORK}")


table = pd.read_csv(find_input(QUERIES_CSV), dtype=str, keep_default_na=False, encoding="utf-8-sig")
for column in ("query_id", "query_vi"):
    if column not in table.columns:
        raise ValueError(f"{QUERIES_CSV} needs column {column!r}")
for column in ("query_en", "objects"):
    if column not in table.columns:
        table[column] = ""
queries = []
for query_id, group in table.groupby("query_id", sort=False):
    for column in ("query_vi", "query_en", "objects"):
        if group[column].nunique() > 1:
            raise ValueError(f"{query_id}: rows disagree on {column}")
    first = group.iloc[0]
    queries.append({
        "query_id": str(query_id),
        "query_vi": first["query_vi"],
        "query_en": first["query_en"],
        "objects": first["objects"],
    })
manifest = json.loads(find_input(MANIFEST_JSON).read_text(encoding="utf-8"))
MODELS = {}
for module in ENCODE:
    entry = manifest["faiss"][module]
    if len(entry.get("model_revisions", [])) != 1:
        raise ValueError(f"{module}: manifest must record exactly one ingest revision")
    MODELS[module] = {
        "model_id": entry["model_ids"][0],
        "model_revision": entry["model_revisions"][0],
        "dimension": entry["dimension"],
    }
print("Runtime:", RUNTIME, "| device:", DEVICE, "| queries:", len(queries))
print(json.dumps(MODELS, indent=2))

## SigLIP 2 text encoder

Chỉ nạp phần text của `google/siglip2-so400m-patch16-384` ở đúng revision trong manifest.
SigLIP 2 được huấn luyện với text viết thường và pad đủ 64 token.

In [ ]:
def encode_siglip(texts):
    from transformers import AutoTokenizer, Siglip2TextModel

    spec = MODELS["siglip"]
    tokenizer = AutoTokenizer.from_pretrained(spec["model_id"], revision=spec["model_revision"])
    model = Siglip2TextModel.from_pretrained(
        spec["model_id"], revision=spec["model_revision"], torch_dtype=torch.float32
    ).to(DEVICE).eval()
    outputs = []
    for start in range(0, len(texts), 32):
        batch = [" ".join(str(text).lower().split()) for text in texts[start:start + 32]]
        inputs = tokenizer(
            batch, padding="max_length", max_length=SIGLIP_MAX_LENGTH,
            truncation=True, return_tensors="pt",
        )
        with torch.inference_mode():
            features = model(input_ids=inputs["input_ids"].to(DEVICE)).pooler_output
        outputs.append(torch.nn.functional.normalize(features.float(), dim=-1).cpu().numpy())
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(outputs)

## BEiT-3 text encoder

Dựng lại đúng như notebook ingest BEiT-3: code UniLM ghim commit, TorchScale 0.2.0 với
shim suy luận, checkpoint được kiểm tra SHA-256 bằng revision trong manifest, tokenizer
`beit3.spm`. Text giữ nguyên hoa/thường như lúc fine-tune COCO, tối đa 64 token.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def download(url, destination, expected_size=None):
    destination = Path(destination)
    if destination.exists() and (expected_size is None or destination.stat().st_size == expected_size):
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    print("Downloading", url)
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for block in response.iter_content(chunk_size=8 * 1024 * 1024):
                stream.write(block)
    if expected_size is not None and partial.stat().st_size != expected_size:
        raise RuntimeError(f"{url}: unexpected size {partial.stat().st_size}")
    partial.replace(destination)
    return destination


def verified(path, expected_sha256):
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise RuntimeError(f"{Path(path).name}: SHA-256 {actual} != {expected_sha256}")
    return Path(path)


def beit3_text_inputs(token_ids, bos, eos, pad, max_length=BEIT3_MAX_LENGTH):
    """Mirror UniLM BaseDataset._get_text_segment: bos + ids + eos, pad; mask 1 on padding."""
    if not token_ids:
        raise ValueError("BEiT-3 query must contain at least one token")
    tokens = [bos, *token_ids[: max_length - 2], eos]
    padding = max_length - len(tokens)
    return tokens + [pad] * padding, [0] * len(tokens) + [1] * padding


def load_beit3():
    from transformers import XLMRobertaTokenizer

    root = MODEL_ROOT / "beit3"
    source_dir = root / "unilm"
    for filename, expected in BEIT3_SOURCE_SHA256.items():
        url = (
            "https://raw.githubusercontent.com/microsoft/unilm/"
            f"{BEIT3_SOURCE_REVISION}/beit3/{filename}"
        )
        verified(download(url, source_dir / filename), expected)
    checkpoint = verified(
        download(
            BEIT3_CHECKPOINT_URL,
            root / "beit3_large_patch16_384_coco_retrieval.pth",
            expected_size=BEIT3_CHECKPOINT_SIZE,
        ),
        MODELS["beit3"]["model_revision"],
    )
    archive = verified(download(TORCHSCALE_URL, root / "torchscale-0.2.0.tar.gz"), TORCHSCALE_SHA256)
    torchscale_root = root / "torchscale-0.2.0"
    if not (torchscale_root / "torchscale" / "__init__.py").exists():
        with tarfile.open(archive, mode="r:gz") as tar:
            tar.extractall(root, filter="data")
    spm = download(BEIT3_TOKENIZER_URL, root / "beit3.spm")

    fairscale = types.ModuleType("fairscale")
    fairscale_nn = types.ModuleType("fairscale.nn")
    fairscale_nn.checkpoint_wrapper = lambda module, *args, **kwargs: module
    fairscale_nn.wrap = lambda module, *args, **kwargs: module
    fairscale.nn = fairscale_nn

    class InferenceOnlyClipLoss(torch.nn.Module):
        def forward(self, *args, **kwargs):
            raise RuntimeError("Training loss is unavailable at query time")

    utils_shim = types.ModuleType("utils")
    utils_shim.ClipLoss = lambda *args, **kwargs: InferenceOnlyClipLoss()
    utils_shim.get_rank = lambda: 0
    utils_shim.get_world_size = lambda: 1
    sys.modules.update({"fairscale": fairscale, "fairscale.nn": fairscale_nn, "utils": utils_shim})
    sys.path[:0] = [str(torchscale_root), str(source_dir)]
    from modeling_finetune import beit3_large_patch16_384_retrieval

    model = beit3_large_patch16_384_retrieval(pretrained=False)
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model.load_state_dict(state.get("model", state), strict=True)
    del state
    return model.to(DEVICE).eval(), XLMRobertaTokenizer(str(spm))


def encode_beit3(texts):
    model, tokenizer = load_beit3()
    outputs = []
    for start in range(0, len(texts), 32):
        batch_tokens, batch_padding = [], []
        for text in texts[start:start + 32]:
            ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(" ".join(str(text).split())))
            tokens, padding = beit3_text_inputs(
                ids, tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id
            )
            batch_tokens.append(tokens)
            batch_padding.append(padding)
        with torch.inference_mode():
            _, language = model(
                text_description=torch.tensor(batch_tokens, device=DEVICE),
                padding_mask=torch.tensor(batch_padding, device=DEVICE),
                only_infer=True,
            )
        outputs.append(torch.nn.functional.normalize(language.float(), dim=-1).cpu().numpy())
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(outputs)

## Encode và xuất file

Query không có text cho một encoder (ví dụ BEiT-3 khi `query_en` trống) nhận một dòng
NaN; phía local sẽ bỏ qua nhánh đó cho query này.

In [ ]:
ENCODERS = {"siglip": encode_siglip, "beit3": encode_beit3}
TEXT_FIELD = {"siglip": "query_vi" if SIGLIP_TEXT == "vi" else "query_en", "beit3": "query_en"}

arrays = {
    field: np.array([query[field] for query in queries], dtype=str)
    for field in ("query_id", "query_vi", "query_en", "objects")
}
for module in ENCODE:
    texts = [query[TEXT_FIELD[module]] for query in queries]
    present = [i for i, text in enumerate(texts) if text.strip()]
    matrix = np.full((len(queries), MODELS[module]["dimension"]), np.nan, dtype=np.float32)
    if present:
        matrix[present] = ENCODERS[module]([texts[i] for i in present])
    norms = np.linalg.norm(matrix[present], axis=1)
    if not np.allclose(norms, 1.0, atol=1e-3):
        raise RuntimeError(f"{module}: vectors are not L2-normalized")
    arrays[module] = matrix
    print(f"{module}: encoded {len(present)}/{len(queries)} queries from {TEXT_FIELD[module]}")

meta = {
    "format_version": FORMAT_VERSION,
    "siglip_text": SIGLIP_TEXT,
    "device": DEVICE,
    "torch": torch.__version__,
    "transformers": installed_version("transformers"),
    "models": {module: MODELS[module] for module in ENCODE},
}
output = WORK / OUTPUT_NPZ
np.savez(output, meta=np.array(json.dumps(meta)), **arrays)
print("Saved", output, f"({output.stat().st_size / 1024:.1f} KiB)")
if RUNTIME == "colab":
    from google.colab import files

    files.download(str(output))
elif RUNTIME == "kaggle":
    print("Download it from the Output panel of this Kaggle notebook.")